# Bluestock Mutual Fund Capstone — Day 1: Data Ingestion & Inital Validation

**Intern:** Adib Azam Shaikh  
**Date:** June 23, 2026  

### Day 1 reflections:
Excited to start the Bluestock Mutual Fund capstone project today! The first step is to ingest all 10 raw datasets. 
I had a small path issue because I didn't set up the directory structure properly (forgot where the raw folder was relative to notebooks), but I fixed it using Path.
Also, tried loading files with typo, and had some weird date parsing blocker. Let's see how that went below.

In [3]:
# Trying to load the file
import pandas as pd
from pathlib import Path
# Oops, I wrote wrong filename to check
df_wrong = pd.read_csv('../data/raw/01_fund_master_wrong.csv')

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/01_fund_master_wrong.csv'

In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path('../data/raw')
datasets = {
    'master': '01_fund_master.csv',
    'nav': '02_nav_history.csv',
    'aum': '03_aum_by_fund_house.csv',
    'sip': '04_monthly_sip_inflows.csv',
    'cat': '05_category_inflows.csv',
    'folio': '06_industry_folio_count.csv',
    'performance': '07_scheme_performance.csv',
    'tx': '08_investor_transactions.csv',
    'holdings': '09_portfolio_holdings.csv',
    'bench': '10_benchmark_indices.csv'
}

dfs = {}
for k, f in datasets.items():
    dfs[k] = pd.read_csv(RAW_DIR / f)
    print(f"Successfully loaded {f}: {dfs[k].shape}")

Successfully loaded 01_fund_master.csv: (40, 15)
Successfully loaded 02_nav_history.csv: (46000, 3)
Successfully loaded 03_aum_by_fund_house.csv: (90, 5)
Successfully loaded 04_monthly_sip_inflows.csv: (48, 6)
Successfully loaded 05_category_inflows.csv: (144, 4)
Successfully loaded 06_industry_folio_count.csv: (21, 5)
Successfully loaded 07_scheme_performance.csv: (40, 15)
Successfully loaded 08_investor_transactions.csv: (32778, 14)
Successfully loaded 09_portfolio_holdings.csv: (322, 9)
Successfully loaded 10_benchmark_indices.csv: (8050, 4)


### Date Parsing Issues
Tried parsing dates as standard DD-MM-YYYY, but some files have month names (e.g., 03-Jan-22). This threw a ValueError!

In [4]:
# Let's convert dates. The date format is DD-MM-YYYY in other files but wait...
pd.to_datetime(dfs['nav']['date'], format='%d-%m-%Y')

ValueError: time data "03-Jan-22" does not match format "%d-%m-%Y" (match)

In [5]:
# Fixed it by trying to catch parsing format differences
try:
    dfs['nav']['parsed_date'] = pd.to_datetime(dfs['nav']['date'], format='%d-%m-%Y')
except Exception:
    dfs['nav']['parsed_date'] = pd.to_datetime(dfs['nav']['date'], format='%d-%b-%Y')
print("Dates converted successfully using fallback format.")

Dates converted successfully using fallback format.


### Live NAV fetch check
Let's fetch live NAVs from api.mfapi.in. I forgot to import requests first!

In [7]:
response = requests.get("https://api.mfapi.in/mf/119551")

NameError: name 'requests' is not defined

In [8]:
import requests
import time

# Now it works
print("Fetching code 119551 (sbi_bluechip)")
res = requests.get("https://api.mfapi.in/mf/119551", timeout=12)
data = res.json()
print(f"  Title: {data['meta']['scheme_name']}")
print(f"  Latest NAV: {data['data'][0]['nav']}")

Fetching code 119551 (sbi_bluechip)
  Title: SBI Bluechip Fund - Regular Plan - Growth
  Latest NAV: 86.42
